In [0]:
from datetime import datetime

In [0]:
jdbc_host_name = "rainbow-server.database.windows.net"
jdbc_database = "rainbow-db"
jdbc_port = 1433
jdbc_url = f"jdbc:sqlserver://{jdbc_host_name}:{jdbc_port};database={jdbc_database}"
jdbc_user = dbutils.secrets.get(scope="sc-rainbow-vault", key="sql-server-user")
jdbc_password = dbutils.secrets.get(scope="sc-rainbow-vault", key="sql-server-password")

In [0]:
# query = "(SELECT TOP (1000) * FROM [SalesLT].[Customer]) as src"
# df = (
#     spark.read.format("jdbc")
#     .option("url", jdbc_url)
#     .option("dbtable", query)
#     .option("user", jdbc_user)
#     .option("password", jdbc_password)
#     .load()
# )
# df.display()

In [0]:
# df.write.format("parquet").save("/mnt/rainbowcontainer/sqlserver/customer")

In [0]:
query = """(select TABLE_SCHEMA, TABLE_NAME from information_schema.tables 
where table_type = 'BASE TABLE') as src
"""
info_df = (
    spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", query)
    .option("user", jdbc_user)
    .option("password", jdbc_password)
    .load()
)

In [0]:
display(info_df)

In [0]:
table_list = info_df.collect()
print(table_list)

In [0]:
for table_info in table_list:
    schema = table_info["TABLE_SCHEMA"]
    table = table_info["TABLE_NAME"]
    target_table = f"{schema}_{table}_{datetime.today().strftime("%Y%m%d")}".lower()
    print(target_table)
    print(f"Loading {schema}.{table}")
    try:
        query = f"(SELECT * FROM {schema}.{table}) as src"
        df = (
            spark.read.format("jdbc")
            .option("url", jdbc_url)
            .option("dbtable", query)
            .option("user", jdbc_user)
            .option("password", jdbc_password)
            .load()
        )
        if table == "BuildVersion":
            df = df.withColumnRenamed("Database Version", "DatabaseVersion")
        df.write.mode("overwrite").save(f"/mnt/rainbowcontainer/sqlserver/{target_table}")
        print(f"Success: {schema}.{table} loaded to {target_table}")
    except Exception as e:
        print(f"Error: {schema}.{table} failed to load")
        print(e)